# Step A1 — Build Adaptive Evaluation Protocol v2

This notebook builds a deterministic point-in-time query set before any new model is trained. It samples multiple strong-positive queries per user across history and timeline strata, assigns exact-target relevance, labels warm/cold targets, sanitizes recent histories, and saves the reusable protocol to Drive. GPU is not required.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from pathlib import Path
from getpass import getpass
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
OUTPUT_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_eval_protocol_v2')
SOURCE_ROOT = WORK_ROOT / 'runtime'
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
print('adaptive root:', ADAPTIVE_ROOT)
print('protocol output:', OUTPUT_DIR)

In [ ]:
required = {
    'pyarrow': 'pyarrow',
    'pandas': 'pandas',
    'huggingface_hub': 'huggingface_hub>=0.32',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import pandas as pd
print('dependencies ready')

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing inputs:\n' + '\n'.join(missing_paths)
for path in required_paths:
    assert (path / '_SUCCESS').exists(), f'Incomplete Spark output: {path}'
print('adaptive data preflight: PASS')

## Download the small protocol runtime

The runtime is downloaded from the private Hugging Face repository. This step does not download the V4 checkpoint. `HF_TOKEN` must be available in Colab Secrets; a secure one-time prompt is used if the Colab secret API times out.

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_EXPECTED_SHA256 = 'dc1a7e9088e4fb7e52685e2249ad5853d88bdc8d0403d51925f3eca54128ecb7'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception as exc:
    print('Colab Secrets unavailable:', type(exc).__name__)
    HF_TOKEN = getpass('Paste Hugging Face READ token: ')
assert HF_TOKEN and HF_TOKEN.startswith('hf_'), 'A valid Hugging Face token must start with hf_.'
runtime_zip = Path(hf_hub_download(
    repo_id=HF_REPO_ID,
    filename='runtime/adaptive_eval_runtime.zip',
    repo_type='model',
    token=HF_TOKEN,
))

def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
builder_script = SOURCE_ROOT / 'scripts/build_adaptive_eval_protocol_v2.py'
assert builder_script.exists(), f'Missing builder: {builder_script}'
print('runtime integrity: PASS')
print('builder:', builder_script)

## Build the reusable query set

The default keeps at most 12 deterministic queries per user. It prioritizes history buckets and early/middle/late timeline coverage, then fills remaining slots evenly.

In [ ]:
command = [
    sys.executable, str(builder_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--split', 'validation',
    '--max-queries-per-user', '12',
    '--seed', '44',
    '--overwrite',
]
log_path = Path('/content/adaptive_eval_protocol_v2.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-120:])
    raise RuntimeError(f'Protocol build failed with exit code {return_code}. Last log lines:\n{tail}')

## Validate protocol coverage

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'manifest.json').read_text())
queries = pd.read_parquet(OUTPUT_DIR / 'queries.parquet')
assert len(queries) == manifest['queries']
assert queries['user_id'].nunique() == manifest['users']
assert queries['example_id'].is_unique
assert queries['relevance_policy'].eq('exact_target_at_query_time').all()
print('queries/users:', len(queries), queries['user_id'].nunique())
print('warm target rate:', manifest['warm_target_rate'])
print('recent repeat target rate:', manifest['recent_repeat_target_rate'])
assert manifest['builder_revision'] == '2.1-session-history'
print('session history distribution:')
display(pd.DataFrame(manifest['session_history_distribution']))
print('global sequence-length distribution (diagnostic only; capped at 30):')
display(pd.DataFrame(manifest['global_sequence_length_distribution']))
display(pd.DataFrame(manifest['timeline_distribution']))
display(queries.drop(columns=['recent_seen_video_ids']).head(10))

## Download the lightweight protocol report

The reusable query Parquet remains organized under the adaptive Gold directory on Drive. The downloaded ZIP contains the manifest, preview, query table, and build log for review.

In [ ]:
archive_path = Path('/content/adaptive_eval_protocol_v2.zip')
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, Path('evaluation_protocol/v2') / path.relative_to(OUTPUT_DIR))
    if log_path.exists():
        archive.write(log_path, 'logs/adaptive_eval_protocol_v2.log')
print('protocol persisted on Drive:', OUTPUT_DIR)
print('download archive:', archive_path, archive_path.stat().st_size, 'bytes')
files.download(str(archive_path))